In [ ]:
import requests
import json
from pathlib import Path
import pandas as pd
from itertools import combinations
import networkx as nx
!pip install GitPython unidecode

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.8/235.8 kB 5.5 MB/s eta 0:00:00


In [ ]:
api_key = "irjTGQKkPBoFg4bRlinSRu"

In [ ]:
work_type = "article"
Path("data").mkdir(exist_ok=True)

In [ ]:
filter_string = (
    f"publication_year:2021-2025,"
    f"institutions.country_code:us|gb|ru|it|br|kr,"  # US, UK, Russia, Italy, Brazil, South Korea
    f"primary_topic.field.id:20,"
    f"type:{work_type},"
    f"is_paratext:false")

print(filter_string)

publication_year:2021-2025,institutions.country_code:us|gb|ru|it|br|kr,primary_topic.field.id:20,type:article,is_paratext:false


In [ ]:
url = "https://api.openalex.org/works"

cursor = "*"
all_works = []
downloaded = 0
total = None
page_num = 0

while True:
    params = {
        "filter": filter_string,
        "per_page": 100,
        "cursor": cursor,
        "api_key": api_key}

    response = requests.get(url, params=params, timeout=60)
    response.raise_for_status()
    data = response.json()

    if total is None:
        total = data["meta"]["count"]

    results = data["results"]
    all_works.extend(results)

    downloaded += len(results)
    page_num += 1

    print(f"\rpages: {page_num} | downloaded: {downloaded} / {total}", end="")

    cursor = data["meta"]["next_cursor"]

    if not results or cursor is None:
        break

print()
print("downloaded works num:", len(all_works))

pages: 1717 | downloaded: 171613 / 171613
downloaded works num: 171613


In [ ]:
countries = {"us", "gb", "ru", "it", "br", "kr"}

In [ ]:
rows = []

for work in all_works:
    authorships = work.get("authorships", [])

    if len(authorships) > 15:
        continue

    paper_id = work.get("id")
    paper_title = work.get("display_name")
    publication_year = work.get("publication_year")

    for authorship in authorships:
        author = authorship.get("author", {})
        author_id = author.get("id")
        author_name = author.get("display_name")

        if author_id is None:
            continue

        institutions = authorship.get("institutions", [])

        all_countries = []
        target_countries = []

        for inst in institutions:
            country_code = inst.get("country_code")

            if country_code is None:
                continue

            country_code = country_code.lower()

            if country_code not in all_countries:
                all_countries.append(country_code)

            if country_code in countries and country_code not in target_countries:
                target_countries.append(country_code)

        rows.append({
            "paper_id": paper_id,
            "paper_title": paper_title,
            "publication_year": publication_year,
            "author_id": author_id,
            "author_name": author_name,
            "author_countries_all": ";".join(sorted(all_countries)),
            "author_countries_target": ";".join(sorted(target_countries)),
            "in_target_countries": int(len(target_countries) > 0)})

authors_df = pd.DataFrame(rows)

In [ ]:
paper_sizes = authors_df.groupby("paper_id")["author_id"].nunique()
print(paper_sizes.describe())
print("papers with 1 author:", (paper_sizes == 1).sum())
print("papers with 2+ authors:", (paper_sizes >= 2).sum())

count    166470.000000
mean          2.912519
std           2.175556
min           1.000000
25%           1.000000
50%           2.000000
75%           4.000000
max          15.000000
Name: author_id, dtype: float64
papers with 1 author: 47218
papers with 2+ authors: 119252


In [ ]:
authors_df.to_csv("data/authors_2021_2025.csv", index = False)

In [ ]:
from itertools import combinations

pair_all = []

for paper_id, group in authors_df.groupby("paper_id"):
    group = group.drop_duplicates(subset="author_id").copy()
    group = group.sort_values("author_id")

    authors = group[[
            "author_id",
            "author_name",
            "author_countries_all",
            "author_countries_target",
            "in_target_countries"]].to_dict("records")

    if len(authors) < 2:
        continue

    for a1, a2 in combinations(authors, 2):
        pair_all.append({
            "paper_id": paper_id,

            "author_1_id": a1["author_id"],
            "author_1_name": a1["author_name"],
            "author_1_countries_all": a1["author_countries_all"],
            "author_1_countries_target": a1["author_countries_target"],
            "author_1_in_target": a1["in_target_countries"],

            "author_2_id": a2["author_id"],
            "author_2_name": a2["author_name"],
            "author_2_countries_all": a2["author_countries_all"],
            "author_2_countries_target": a2["author_countries_target"],
            "author_2_in_target": a2["in_target_countries"]})

pairs_df = pd.DataFrame(pair_all)

print("len(pairs_df):", len(pairs_df))
print("unique papers in pairs:", pairs_df["paper_id"].nunique())
pairs_df.head()

len(pairs_df): 857592
unique papers in pairs: 119252


,paper_id,author_1_id,author_1_name,author_1_countries_all,author_1_countries_target,author_1_in_target,author_2_id,author_2_name,author_2_countries_all,author_2_countries_target,author_2_in_target
0,https://openalex.org/W116056414,https://openalex.org/A5011711127,Miriam Sturkenboom,nl,,0,https://openalex.org/A5045075334,Giovanni Corrao,it,it,1
1,https://openalex.org/W116056414,https://openalex.org/A5011711127,Miriam Sturkenboom,nl,,0,https://openalex.org/A5070504206,Silvana Romio,it;nl,it,1
2,https://openalex.org/W116056414,https://openalex.org/A5045075334,Giovanni Corrao,it,it,1,https://openalex.org/A5070504206,Silvana Romio,it;nl,it,1
3,https://openalex.org/W127661761,https://openalex.org/A5015249696,Rosanna Tarricone,it,it,1,https://openalex.org/A5032214375,Marta Gehring,ch,,0
4,https://openalex.org/W127661761,https://openalex.org/A5015249696,Rosanna Tarricone,it,it,1,https://openalex.org/A5036176228,Mariapia Cirenei,it,it,1


In [ ]:
print(len(pairs_df))
print(pairs_df["paper_id"].nunique())

857592
119252


In [ ]:
pairs_df.to_csv("data/pairs_2021_2025.csv", index = False)
print("saved")

saved


In [ ]:
pairs_focal = pairs_df[(pairs_df["author_1_in_target"] == 1) | (pairs_df["author_2_in_target"] == 1)].copy()

edges_df = (pairs_focal.groupby(["author_1_id", "author_2_id"], as_index=False)
    .size().rename(columns={"size": "weight"}))

edges_df = edges_df[edges_df["weight"] <= 15].copy()

print("edges:", len(edges_df))
print("sum weight:", edges_df["weight"].sum())
print("max weight:", edges_df["weight"].max())
edges_df.head()


edges: 585090
sum weight: 689759
max weight: 15


,author_1_id,author_2_id,weight
0,https://openalex.org/A5000000076,https://openalex.org/A5063275369,1
1,https://openalex.org/A5000000572,https://openalex.org/A5000865376,1
2,https://openalex.org/A5000000572,https://openalex.org/A5004773077,3
3,https://openalex.org/A5000000572,https://openalex.org/A5015162351,1
4,https://openalex.org/A5000000572,https://openalex.org/A5020897161,1


In [ ]:
print(len(edges_df))
print(edges_df["weight"].sum())
print(edges_df["weight"].max())
print((edges_df["weight"] == 1).sum())

585090
689759
15
518357


In [ ]:
authors_df = authors_df.drop_duplicates(subset=["paper_id", "author_id"]).copy()

In [ ]:
node_rows = []

for author_id, group in authors_df.groupby("author_id"):
    author_name = group["author_name"].iloc[0]
    num_papers = group["paper_id"].nunique()
    in_target_countries = int(group["in_target_countries"].max())

    all_country_counts = {}
    target_country_counts = {}

    for x in group["author_countries_all"].dropna():
        if x != "":
            for c in x.split(";"):
                all_country_counts[c] = all_country_counts.get(c, 0) + 1

    for x in group["author_countries_target"].dropna():
        if x != "":
            for c in x.split(";"):
                target_country_counts[c] = target_country_counts.get(c, 0) + 1

    all_countries = sorted(all_country_counts.keys())
    target_countries = sorted(target_country_counts.keys())

    main_country_all = max(all_country_counts, key=all_country_counts.get) if all_country_counts else pd.NA
    main_country_target = max(target_country_counts, key=target_country_counts.get) if target_country_counts else pd.NA

    node_rows.append({
        "author_id": author_id,
        "author_name": author_name,
        "author_countries_all": ";".join(all_countries),
        "author_countries_target": ";".join(target_countries),
        "in_target_countries": in_target_countries,
        "main_country_all": main_country_all,
        "main_country_target": main_country_target,
        "num_papers": num_papers})

nodes_df = pd.DataFrame(node_rows)
nodes_df.head()

,author_id,author_name,author_countries_all,author_countries_target,in_target_countries,main_country_all,main_country_target,num_papers
0,https://openalex.org/A5000000076,I. M. Shor,ru,ru,1,ru,ru,1
1,https://openalex.org/A5000000572,Piotr Faliszewski,pl;us,us,1,pl,us,9
2,https://openalex.org/A5000001505,Lawrence M. Vielhaber,us,us,1,us,us,1
3,https://openalex.org/A5000001782,Nancy Birdsall,us,us,1,us,us,1
4,https://openalex.org/A5000002187,Kafilah Lola Gold,ng;za,,0,ng,<NA>,1


In [ ]:
print("n of nodes:", len(nodes_df))

n of nodes: 294662


In [ ]:
nodes_df["main_country_target"].value_counts()

,count
main_country_target,
us,115477
gb,32998
ru,25064
br,18119
it,14477
kr,7058


In [ ]:
nodes_df.to_csv("data/nodes_2021_2025.csv", index=False)
print("saved")

saved


In [ ]:
g = nx.Graph()

for _, row in edges_df.iterrows():
    g.add_edge(row["author_1_id"], row["author_2_id"], weight=row["weight"])

print("nodes:", g.number_of_nodes())
print("edges:", g.number_of_edges())

nodes: 266069
edges: 585090


In [ ]:
g.number_of_edges() == len(edges_df)

True

In [ ]:
print("len(edges_df):", len(edges_df))

dup_pairs = edges_df.duplicated(subset=["author_1_id", "author_2_id"]).sum()
print("duplicate pairs by ids only:", dup_pairs)

len(edges_df): 585090
duplicate pairs by ids only: 0


In [ ]:
paper_sizes = authors_df.groupby("paper_id")["author_id"].nunique()

print(paper_sizes.describe())

print("papers with 1 author in filtered data:", (paper_sizes == 1).sum())
print("papers with 2+ authors in filtered data:", (paper_sizes >= 2).sum())
print("mean team size:", paper_sizes.mean())

count    166470.000000
mean          2.912519
std           2.175556
min           1.000000
25%           1.000000
50%           2.000000
75%           4.000000
max          15.000000
Name: author_id, dtype: float64
papers with 1 author in filtered data: 47218
papers with 2+ authors in filtered data: 119252
mean team size: 2.9125187721511385


In [ ]:
print("average degree:", 2 * g.number_of_edges() / g.number_of_nodes())
print("connected components:", nx.number_connected_components(g))

component_sizes = sorted((len(c) for c in nx.connected_components(g)), reverse=True)
print("largest component:", component_sizes[0])
print("top 10 component sizes:", component_sizes[:10])

average degree: 4.398032089420414
connected components: 33939
largest component: 130592
top 10 component sizes: [130592, 176, 173, 129, 108, 84, 75, 70, 67, 57]


In [ ]:
components = list(nx.connected_components(g))

print("len(components)", len(components))
print("len(max(components))", len(max(components, key=len)))

len(components) 33939
len(max(components)) 130592


In [ ]:
degree_dict = dict(g.degree())
weighted_degree_dict = dict(g.degree(weight="weight"))

In [ ]:
nodes_df["degree"] = nodes_df["author_id"].map(degree_dict).fillna(0).astype(int)
nodes_df["weighted_degree"] = nodes_df["author_id"].map(weighted_degree_dict).fillna(0).astype(int)

nodes_df.head()

,author_id,author_name,author_countries_all,author_countries_target,in_target_countries,main_country_all,main_country_target,num_papers,degree,weighted_degree
0,https://openalex.org/A5000000076,I. M. Shor,ru,ru,1,ru,ru,1,1,1
1,https://openalex.org/A5000000572,Piotr Faliszewski,pl;us,us,1,pl,us,9,18,25
2,https://openalex.org/A5000001505,Lawrence M. Vielhaber,us,us,1,us,us,1,0,0
3,https://openalex.org/A5000001782,Nancy Birdsall,us,us,1,us,us,1,3,3
4,https://openalex.org/A5000002187,Kafilah Lola Gold,ng;za,,0,ng,<NA>,1,1,1


In [ ]:
nodes_target = nodes_df[nodes_df["in_target_countries"] == 1].copy()

In [ ]:
largest_component = max(nx.connected_components(g), key=len)
largest_component_set = set(largest_component)

nodes_target["is_in_giant_component"] = nodes_target["author_id"].isin(largest_component_set)

In [ ]:
print(nodes_target[["num_papers", "degree", "weighted_degree"]].describe())

          num_papers         degree  weighted_degree
count  213193.000000  213193.000000    213193.000000
mean        1.772999       4.703546         5.593242
std         2.511682       5.999460         9.233626
min         1.000000       0.000000         0.000000
25%         1.000000       1.000000         1.000000
50%         1.000000       3.000000         3.000000
75%         2.000000       6.000000         7.000000
max       171.000000     245.000000       500.000000


In [ ]:
nodes_target["is_in_giant_component"].value_counts()

,count
is_in_giant_component,
False,124189
True,89004


In [ ]:
nodes_df.to_csv("data/nodes_2021_2025_with_metrics.csv", index=False)
nodes_target.to_csv("data/nodes_target_2021_2025_with_metrics.csv", index=False)
print("saved")

saved


In [ ]:
nodes_target.groupby("main_country_target")[["num_papers", "degree", "weighted_degree"]].mean()

,num_papers,degree,weighted_degree
main_country_target,,,
br,1.461173,3.597881,4.028423
gb,1.881841,4.902600,5.826383
it,2.057678,5.092284,6.219590
kr,1.829980,3.625106,4.247662
ru,1.478296,2.205634,2.482844
us,1.815617,5.379495,6.450973


In [ ]:
nodes_target.groupby("main_country_target")[["num_papers", "degree", "weighted_degree"]].agg(["mean", "median", "std"])

num_papers                     degree                   \
                          mean median       std      mean median       std   
main_country_target                                                          
br                    1.461173    1.0  1.523928  3.597881    3.0  3.658421   
gb                    1.881841    1.0  2.645003  4.902600    3.0  6.409955   
it                    2.057678    1.0  2.656654  5.092284    4.0  5.555485   
kr                    1.829980    1.0  3.458288  3.625106    3.0  4.377924   
ru                    1.478296    1.0  2.208013  2.205634    2.0  3.238724   
us                    1.815617    1.0  2.561232  5.379495    4.0  6.571864   

                    weighted_degree                    
                               mean median        std  
main_country_target                                    
br                         4.028423    3.0   4.818097  
gb                         5.826383    3.0   9.366833  
it                         6.219590    4.0   8.055250  
kr                         4.247662    3.0   7.035035  
ru                         2.482844    2.0   4.828820  
us                         6.450973    4.0  10.444093

In [ ]:
nodes_target.groupby("main_country_target")["is_in_giant_component"].mean()

,is_in_giant_component
main_country_target,
br,0.265909
gb,0.463361
it,0.498929
kr,0.374044
ru,0.110278
us,0.487275


In [ ]:
print((nodes_target["degree"] == 0).sum())

27709


The following code cells as well as this method of extracting gender from name are taken from https://github.com/IES-platform/r4r_gender/tree/main

In [ ]:
import requests
import zipfile
from io import BytesIO
import os
import sys

url = "https://github.com/ClemSternWIPO/gender_it/archive/refs/heads/main.zip"

r = requests.get(url)
r.raise_for_status()

z = zipfile.ZipFile(BytesIO(r.content))
z.extractall("gender_it_local")

print(os.listdir("gender_it_local"))

['gender_it-main']


In [ ]:
sys.path.append("gender_it_local/gender_it-main")
import gender_it_functions as gf

In [ ]:
import re
import numpy as np

def extract_first_name(full_name):
    if pd.isna(full_name):
        return np.nan

    name = str(full_name).strip()
    name = re.sub(r"\s+", " ", name)

    if name == "":
        return np.nan

    first = name.split(" ")[0].strip()
    first_clean = first.replace(".", "")

    if len(first_clean) <= 1:
        return np.nan

    if not re.search(r"[A-Za-zÀ-ÿĀ-žА-Яа-я]", first_clean):
        return np.nan

    return first_clean

In [ ]:
gender_input = (nodes_df[["author_id", "author_name", "main_country_all"]]
    .drop_duplicates(subset=["author_id"])
    .rename(columns={
        "author_name": "name",
        "main_country_all": "country_code"}).copy())

gender_input["first_name"] = gender_input["name"].apply(extract_first_name)
gender_input.head()

,author_id,name,country_code,first_name
0,https://openalex.org/A5000000076,I. M. Shor,ru,NaN
1,https://openalex.org/A5000000572,Piotr Faliszewski,pl,Piotr
2,https://openalex.org/A5000001505,Lawrence M. Vielhaber,us,Lawrence
3,https://openalex.org/A5000001782,Nancy Birdsall,us,Nancy
4,https://openalex.org/A5000002187,Kafilah Lola Gold,ng,Kafilah


In [ ]:
print("missing first_name:", gender_input["first_name"].isna().sum())

missing first_name: 18837


In [ ]:
gender_input[gender_input["first_name"].isna()]["country_code"].value_counts()

,count
country_code,
ru,5790
us,4950
gb,1671
it,616
br,371
...,...
si,1
gy,1
sy,1


In [ ]:
gender_input[gender_input["first_name"].isna()].sample(30)[["name", "country_code"]]

,name,country_code
291878,L. V. Romanenko,us
142275,L Dionisi,it
185225,A. V. Bosov,ru
273796,J. Costello,<NA>
114406,A Myint Zu,us
293513,T. V. Altunin,us
275129,M Fraser,us
103967,A. N. Shadrintseva,ru
74885,G. O. Zhanguttina,kz
236097,H. J. Yang,cn


In [ ]:
dict_path = "gender_it_local/gender_it-main/dictionaries/"

In [ ]:
d2_1 = pd.read_csv(dict_path + "d2_1.csv.gz", compression="gzip")
d2_2 = pd.read_csv(dict_path + "d2_2.csv.gz", compression="gzip")
d2_3 = pd.read_csv(dict_path + "d2_3.csv.gz", compression="gzip")

d2 = pd.concat([d2_1, d2_2, d2_3], ignore_index=True)

d2.to_csv(dict_path + "d2.csv.gz", index=False, compression="gzip")

print("saved:", dict_path + "d2.csv.gz")
print(d2.shape)

saved: gender_it_local/gender_it-main/dictionaries/d2.csv.gz
(26043223, 3)


In [ ]:
import os
print(os.listdir(dict_path))

['d2_3.csv.gz', 'd3.csv.gz', 'd2_1.csv.gz', 'd1.csv.gz', 'Data description', 'd2_2.csv.gz', 'd2.csv.gz']


In [ ]:
gender_work = gender_input.dropna(subset=["first_name", "country_code"])[["author_id", "first_name", "country_code"]].copy()
gender_work = gender_work.rename(columns={"first_name": "name"})

gendered_nodes = gf.get_gender(gender_work, name_column="name", country_column="country_code", threshold=0.75, path=dict_path)

Step 1 - Reading the name-country-gender dictionary
reading the dictionnary.
Step 2 - Reading the name-language-gender dictionary
reading the dictionnary.
Step 3 - Reading the name-gender dictionary
reading the dictionnary.
dff         name_id clean_name  surname_position clean_country_column
218812   218812      huili                 2                   US
Results distribution is as follows:
             count  Percentage
gender                       
M          131928   51.123589
F           90054   34.896941
not found   34400   13.330388
?            1675    0.649081


In [ ]:
print(gendered_nodes.columns.tolist())
gendered_nodes.head()

['level', 'gender', '?', 'F', 'M', 'author_id', 'name', 'country_code']


,level,gender,?,F,M,author_id,name,country_code
0,1,M,0.0,0.000000,1.0,https://openalex.org/A5000000572,Piotr,pl
1,2,M,0.0,0.000000,1.0,https://openalex.org/A5000001505,Lawrence,us
2,1,F,0.0,0.997062,0.0,https://openalex.org/A5000001782,Nancy,us
3,3,not found,0.0,0.000000,0.0,https://openalex.org/A5000002187,Kafilah,ng
4,2,M,0.0,0.000000,1.0,https://openalex.org/A5000003309,David,us


In [ ]:
gendered_nodes["name_inferred_gender"] = gendered_nodes["gender"].replace({"F": "female", "M": "male", "?": "unknown", "not found": "unknown"})

In [ ]:
nodes_df = nodes_df.merge(gendered_nodes[["author_id", "gender",
                                          "name_inferred_gender", "F", "M", "level"]],
                          on="author_id", how="left")

nodes_df["name_inferred_gender"] = nodes_df["name_inferred_gender"].fillna("unknown")
nodes_df["gender"] = nodes_df["gender"].fillna("not found")

In [ ]:
nodes_target = nodes_df[nodes_df["in_target_countries"] == 1].copy()

In [ ]:
nodes_df["name_inferred_gender"].value_counts(dropna=False)

,count
name_inferred_gender,
male,131928
female,90054
unknown,72680


In [ ]:
pd.crosstab(nodes_df["level"], nodes_df["name_inferred_gender"], dropna=False)

name_inferred_gender,female,male,unknown
level,,,
1.0,68563,60216,1675
2.0,8380,53586,0
3.0,13111,18126,34400
NaN,0,0,36605


In [ ]:
nodes_df.to_csv("data/nodes_2021_2025_with_gender.csv", index=False)
nodes_target.to_csv("data/nodes_target_2021_2025_with_gender.csv", index=False)
print("saved")

saved


In [ ]:
pd.crosstab(nodes_df["main_country_target"], nodes_df["name_inferred_gender"])

name_inferred_gender,female,male,unknown
main_country_target,,,
br,5992,9909,2218
gb,10981,17395,4622
it,5155,8346,976
kr,1098,1515,4445
ru,8354,7339,9371
us,41040,57553,16884


In [ ]:
pd.crosstab(nodes_df["main_country_target"], nodes_df["name_inferred_gender"], normalize="index")

name_inferred_gender,female,male,unknown
main_country_target,,,
br,0.330703,0.546884,0.122413
gb,0.332778,0.527153,0.140069
it,0.356082,0.576501,0.067417
kr,0.155568,0.214650,0.629782
ru,0.333307,0.292810,0.373883
us,0.355395,0.498394,0.146211


In [ ]:
analysis_df = nodes_df[nodes_df["name_inferred_gender"].isin(["female", "male"])].copy()

analysis_df.groupby("name_inferred_gender")[["num_papers", "degree", "weighted_degree"]].agg(["mean", "median", "std"])

num_papers                     degree                   \
                           mean median       std      mean median       std   
name_inferred_gender                                                          
female                 1.530482    1.0  1.775171  4.445588    3.0  5.584924   
male                   1.860522    1.0  2.650754  4.188171    2.0  5.780250   

                     weighted_degree                   
                                mean median       std  
name_inferred_gender                                   
female                      5.180603    3.0  8.673069  
male                        5.066514    3.0  8.756974

In [ ]:
analysis_df = nodes_target[nodes_target["name_inferred_gender"].isin(["female", "male"])].copy()

analysis_df.groupby("name_inferred_gender")[["num_papers", "degree", "weighted_degree"]].agg(["mean", "median", "std"])

num_papers                     degree                   \
                           mean median       std      mean median       std   
name_inferred_gender                                                          
female                 1.591159    1.0  1.919565  5.031630    3.0  6.001861   
male                   1.975935    1.0  2.898148  4.808293    3.0  6.345917   

                     weighted_degree                   
                                mean median       std  
name_inferred_gender                                   
female                      5.890072    4.0  9.447149  
male                        5.844989    3.0  9.697964

In [ ]:
analysis_df.groupby(["main_country_target", "name_inferred_gender"])[["num_papers", "degree", "weighted_degree"]].agg(["mean", "median", "std"])

num_papers                   \
                                               mean median       std   
main_country_target name_inferred_gender                               
br                  female                 1.294226    1.0  1.040033   
                    male                   1.584822    1.0  1.803978   
gb                  female                 1.645934    1.0  1.855023   
                    male                   2.105145    1.0  3.142076   
it                  female                 1.786227    1.0  1.932325   
                    male                   2.311526    1.0  3.101441   
kr                  female                 1.545537    1.0  1.451188   
                    male                   1.953135    1.0  3.896250   
ru                  female                 1.417405    1.0  1.468595   
                    male                   1.634964    1.0  3.134860   
us                  female                 1.631944    1.0  2.109756   
                    male                   1.999635    1.0  2.870536   

                                            degree                   \
                                              mean median       std   
main_country_target name_inferred_gender                              
br                  female                3.772196    3.0  3.533327   
                    male                  3.489858    2.0  3.742712   
gb                  female                5.076951    3.0  6.192730   
                    male                  4.928773    3.0  6.837411   
it                  female                5.007177    4.0  5.027220   
                    male                  5.080518    3.0  5.951068   
kr                  female                3.526412    3.0  3.561638   
                    male                  3.706271    2.0  4.989043   
ru                  female                2.274240    2.0  2.528580   
                    male                  2.296907    1.0  4.495994   
us                  female                5.808017    4.0  6.663418   
                    male                  5.308655    3.0  6.715806   

                                         weighted_degree                    
                                                    mean median        std  
main_country_target name_inferred_gender                                    
br                  female                      4.105975    3.0   4.351285  
                    male                        3.994550    3.0   5.120881  
gb                  female                      5.880066    4.0   8.755446  
                    male                        6.012130    3.0  10.250985  
it                  female                      5.922017    4.0   7.037935  
                    male                        6.426432    4.0   8.857363  
kr                  female                      3.942623    3.0   4.422676  
                    male                        4.390099    3.0   7.713515  
ru                  female                      2.491381    2.0   3.183706  
                    male                        2.709361    2.0   7.227158  
us                  female                      6.893153    4.0  11.041836  
                    male                        6.466891    4.0  10.419241

In [ ]:
strict_df = nodes_df[(nodes_df["name_inferred_gender"].isin(["female", "male"])) & (nodes_df["level"].isin([1, 2]))].copy()

In [ ]:
strict_df.groupby(["main_country_target", "name_inferred_gender"])[["num_papers", "degree", "weighted_degree"]].agg(["mean", "median", "std"])

num_papers                   \
                                               mean median       std   
main_country_target name_inferred_gender                               
br                  female                 1.287513    1.0  0.983848   
                    male                   1.591885    1.0  1.736145   
gb                  female                 1.635580    1.0  1.834709   
                    male                   2.108133    1.0  3.156487   
it                  female                 1.794205    1.0  1.946159   
                    male                   2.322019    1.0  3.115146   
kr                  female                 1.634877    1.0  1.589819   
                    male                   2.188776    1.0  4.601690   
ru                  female                 1.446463    1.0  1.483239   
                    male                   1.648389    1.0  2.837935   
us                  female                 1.634546    1.0  2.118508   
                    male                   2.012516    1.0  2.896052   

                                            degree                   \
                                              mean median       std   
main_country_target name_inferred_gender                              
br                  female                3.786464    3.0  3.611775   
                    male                  3.474187    2.0  3.662710   
gb                  female                5.141556    4.0  6.252848   
                    male                  4.972460    3.0  6.890109   
it                  female                5.011312    4.0  5.011071   
                    male                  5.101091    3.0  5.985866   
kr                  female                4.016349    3.0  3.973925   
                    male                  4.307823    3.0  5.280107   
ru                  female                2.311850    2.0  2.595864   
                    male                  2.244971    1.0  3.427799   
us                  female                5.916247    4.0  6.754706   
                    male                  5.371645    4.0  6.809772   

                                         weighted_degree                    
                                                    mean median        std  
main_country_target name_inferred_gender                                    
br                  female                      4.118573    3.0   4.410943  
                    male                        3.973245    3.0   4.927786  
gb                  female                      5.948766    4.0   8.818518  
                    male                        6.071187    3.0  10.343835  
it                  female                      5.941060    4.0   7.049779  
                    male                        6.459993    4.0   8.901980  
kr                  female                      4.465940    3.0   4.727932  
                    male                        5.267007    3.0   9.441389  
ru                  female                      2.546360    2.0   3.294022  
                    male                        2.637351    2.0   5.462712  
us                  female                      7.031642    5.0  11.228172  
                    male                        6.554523    4.0  10.569511

### Gender composition of edges

In [ ]:
edge_gender_df = edges_df.copy()

gender_smth = (nodes_df[["author_id", "name_inferred_gender"]].drop_duplicates(subset=["author_id"])
    .rename(columns={"name_inferred_gender": "gender"}))

edge_gender_df = edge_gender_df.merge(gender_smth.rename(columns={"author_id": "author_1_id", "gender": "gender_1"}),
    on="author_1_id", how="left")

edge_gender_df = edge_gender_df.merge(gender_smth.rename(columns={"author_id": "author_2_id", "gender": "gender_2"}),
    on="author_2_id", how="left")

edge_gender_df[["author_1_id", "gender_1", "author_2_id", "gender_2", "weight"]].head()

,author_1_id,gender_1,author_2_id,gender_2,weight
0,https://openalex.org/A5000000076,unknown,https://openalex.org/A5063275369,unknown,1
1,https://openalex.org/A5000000572,male,https://openalex.org/A5000865376,male,1
2,https://openalex.org/A5000000572,male,https://openalex.org/A5004773077,male,3
3,https://openalex.org/A5000000572,male,https://openalex.org/A5015162351,male,1
4,https://openalex.org/A5000000572,male,https://openalex.org/A5020897161,male,1


In [ ]:
def edge_gender(g1, g2):
    if pd.isna(g1) or pd.isna(g2):
        return "unknown"

    if g1 == "unknown" or g2 == "unknown":
        return "unknown"

    if g1 == "female" and g2 == "female":
        return "female-female"
    if g1 == "male" and g2 == "male":
        return "male-male"

    if {g1, g2} == {"female", "male"}:
        return "female-male"

    return "unknown"


edge_gender_df["edge_gender_type"] = edge_gender_df.apply(lambda row: edge_gender(row["gender_1"], row["gender_2"]),axis=1)

In [ ]:
edge_gender_counts = edge_gender_df["edge_gender_type"].value_counts().rename_axis("edge_gender_type").reset_index(name="num_edges")
edge_gender_counts

,edge_gender_type,num_edges
0,unknown,182912
1,female-male,167480
2,male-male,149350
3,female-female,85348


In [ ]:
print(nodes_df["name_inferred_gender"].value_counts(dropna=False))
(nodes_df["name_inferred_gender"] == "unknown").mean()
print((edge_gender_df["gender_1"] == "unknown").mean())
print((edge_gender_df["gender_2"] == "unknown").mean())
print((edge_gender_df["edge_gender_type"] == "unknown").mean())

name_inferred_gender
male       131928
female      90054
unknown     72680
Name: count, dtype: int64
0.17848194294894804
0.19291391068040814
0.3126219897793502


In [ ]:
c_edge_gender_df = edge_gender_df[edge_gender_df["edge_gender_type"] != "unknown"].copy()

In [ ]:
c_edge_stats = (c_edge_gender_df.groupby("edge_gender_type", as_index=False)
                .agg(num_edges=("weight", "size"),total_weight=("weight", "sum")))

c_edge_stats["share_edges"] = c_edge_stats["num_edges"] / c_edge_stats["num_edges"].sum()
c_edge_stats["share_weight"] = c_edge_stats["total_weight"] / c_edge_stats["total_weight"].sum()

c_edge_stats = c_edge_stats.sort_values("num_edges", ascending=False)
c_edge_stats

,edge_gender_type,num_edges,total_weight,share_edges,share_weight
1,female-male,167480,198217,0.416433,0.410091
2,male-male,149350,185481,0.371353,0.383741
0,female-female,85348,99651,0.212214,0.206168


In [ ]:
edge_gender_df.to_csv("data/edges_2021_2025_with_gender.csv", index=False)
c_edge_stats.to_csv("data/edge_gender_stats_2021_2025_without_NaN.csv", index=False)

### By country

In [ ]:
edge_country_df = c_edge_gender_df.copy()
country_lookup = nodes_df[["author_id", "main_country_target"]].copy()

In [ ]:
edge_country_df = edge_country_df.merge(country_lookup.rename(columns={"author_id": "author_1_id", "main_country_target": "country_1"}),
                                        on="author_1_id",how="left")

In [ ]:
edge_country_df = edge_country_df.merge(country_lookup.rename(columns={"author_id": "author_2_id", "main_country_target": "country_2"}),
                                        on="author_2_id",how="left")

In [ ]:
within_country_edges = edge_country_df[edge_country_df["country_1"] == edge_country_df["country_2"]].copy()
within_country_edges["country"] = within_country_edges["country_1"]

In [ ]:
country_edge_stats = (within_country_edges.groupby(["country", "edge_gender_type"], as_index=False)
                             .agg(num_edges=("weight", "size"),total_weight=("weight", "sum")))

In [ ]:
country_edge_stats["share_edges"] = (country_edge_stats["num_edges"] / country_edge_stats.groupby("country")["num_edges"].transform("sum"))

country_edge_stats["share_weight"] = (country_edge_stats["total_weight"] / country_edge_stats.groupby("country")["total_weight"].transform("sum"))

country_edge_stats = country_edge_stats.sort_values(["country", "num_edges"], ascending=[True, False])

country_edge_table = country_edge_stats.pivot(index="country", columns="edge_gender_type",values="share_edges").fillna(0)

country_edge_table

edge_gender_type,female-female,female-male,male-male
country,,,
br,0.198176,0.401652,0.400172
gb,0.209753,0.423186,0.367061
it,0.182596,0.426574,0.390829
kr,0.246414,0.394093,0.359494
ru,0.354157,0.427750,0.218093
us,0.243383,0.422627,0.333990


### gender assortativity

In [ ]:
gender_attr = nodes_df.set_index("author_id")["name_inferred_gender"].to_dict()
country_attr = nodes_df.set_index("author_id")["main_country_target"].to_dict()

nx.set_node_attributes(g, gender_attr, "gender")
nx.set_node_attributes(g, country_attr, "country")

In [ ]:
classified_nodes = [n for n, d in g.nodes(data=True)
                    if d.get("gender") in ["female", "male"]]

g_classified = g.subgraph(classified_nodes).copy()

In [ ]:
overall_gender_assortativity = nx.attribute_assortativity_coefficient(g_classified, "gender")
overall_gender_assortativity

0.1454945489844511

In [ ]:
country_assort_rows = []

for country in sorted(nodes_df["main_country_target"].dropna().unique()):
    country_nodes = [n for n, d in g_classified.nodes(data=True)
                     if d.get("country") == country]

    subg = g_classified.subgraph(country_nodes).copy()

    if subg.number_of_edges() == 0:
        assort = float("nan")
    else:
        assort = nx.attribute_assortativity_coefficient(subg, "gender")

    country_assort_rows.append({"country": country,
                                "num_nodes": subg.number_of_nodes(),
                                "num_edges": subg.number_of_edges(),
                                "gender_assortativity": assort})

country_assort_df = pd.DataFrame(country_assort_rows)

In [ ]:
country_assort_df

,country,num_nodes,num_edges,gender_assortativity
0,br,14289,20941,0.162524
1,gb,24132,38307,0.132152
2,it,11949,23248,0.108181
3,kr,2365,1185,0.201605
4,ru,12168,10789,0.128362
5,us,87688,200581,0.147749


In [ ]:
nodes_df.to_csv("data/nodes_2024_final.csv", index=False)
edges_df.to_csv("data/edges_2024.csv", index=False)
edge_gender_df.to_csv("data/edges_2024_with_gender.csv", index=False)
c_edge_stats.to_csv("data/edge_gender_stats_2024_classified.csv", index=False)
country_edge_stats.to_csv("data/country_edge_stats_2024.csv", index=False)
country_edge_table.to_csv("data/country_edge_share_table_2024.csv")
obs_exp_country.to_csv("data/obs_exp_country_2024.csv")
country_assort_df.to_csv("data/country_gender_assortativity_2024.csv", index=False)

RANDOM MODEL

In [ ]:
obs_assort = nx.attribute_assortativity_coefficient(g_classified, "gender")
print(obs_assort)

0.1454945489844511


In [ ]:
g_rand = g_classified.copy()

nx.double_edge_swap(g_rand,
    nswap=10 * g_rand.number_of_edges(),
    max_tries=100 * g_rand.number_of_edges())

rand_assort = nx.attribute_assortativity_coefficient(g_rand, "gender")
print(rand_assort)

0.0007163863500043972


In [ ]:
B = 100
null_vals = []

nodes = list(g_classified.nodes())
genders = np.array([g_classified.nodes[n]["gender"] for n in nodes])

for _ in range(B):
    shuffled = np.random.permutation(genders)

    g_rand = g_classified.copy()
    nx.set_node_attributes(g_rand,
        {n: shuffled[i] for i, n in enumerate(nodes)},
        "gender")

    val = nx.attribute_assortativity_coefficient(g_rand, "gender")
    null_vals.append(val)

null_vals = np.array(null_vals)

print("observed:", obs_assort)
print("null mean:", null_vals.mean())
print("null std:", null_vals.std(ddof=1))
print("z-score:", (obs_assort - null_vals.mean()) / null_vals.std(ddof=1))

observed: 0.1454945489844511
null mean: 4.788735769675291e-05
null std: 0.0014951239508709962
z-score: 97.28067130623067


In [ ]:
def edge_type(g1, g2):
    if g1 == "female" and g2 == "female":
        return "female-female"
    elif g1 == "male" and g2 == "male":
        return "male-male"
    else:
        return "female-male"

obs_counts = {"female-female": 0, "male-male": 0, "female-male": 0}

for u, v in g_classified.edges():
    t = edge_type(g_classified.nodes[u]["gender"], g_classified.nodes[v]["gender"])
    obs_counts[t] += 1

obs_counts

{'female-female': 85348, 'male-male': 149350, 'female-male': 167480}

In [ ]:

B = 100
nodes = list(g_classified.nodes())
genders = np.array([g_classified.nodes[n]["gender"] for n in nodes])

null_results = []

for _ in range(B):
    shuffled = np.random.permutation(genders)

    g_rand = g_classified.copy()
    nx.set_node_attributes(g_rand,
        {n: shuffled[i] for i, n in enumerate(nodes)},
        "gender")

    rand_counts = {"female-female": 0, "male-male": 0, "female-male": 0}

    for u, v in g_rand.edges():
        t = edge_type(g_rand.nodes[u]["gender"], g_rand.nodes[v]["gender"])
        rand_counts[t] += 1

    null_results.append(rand_counts)

null_df = pd.DataFrame(null_results)

result_df = pd.DataFrame({
    "edge_type": ["female-female", "female-male", "male-male"],
    "observed": [obs_counts["female-female"],
        obs_counts["female-male"],
        obs_counts["male-male"]],
    "random_mean": [null_df["female-female"].mean(),
        null_df["female-male"].mean(),
        null_df["male-male"].mean()],
    "random_sd": [null_df["female-female"].std(ddof=1),
        null_df["female-male"].std(ddof=1),
        null_df["male-male"].std(ddof=1)]})

result_df["difference"] = result_df["observed"] - result_df["random_mean"]
result_df["pct_diff"] = 100 * result_df["difference"] / result_df["random_mean"]

result_df.round(2)

,edge_type,observed,random_mean,random_sd,difference,pct_diff
0,female-female,85348,67279.18,515.36,18068.82,26.86
1,female-male,167480,194443.66,363.78,-26963.66,-13.87
2,male-male,149350,140455.16,730.67,8894.84,6.33


In [ ]:
import matplotlib.pyplot as plt


gender_lookup = nx.get_node_attributes(g_classified, "gender")

paper_year = authors_df[["paper_id", "publication_year"]].drop_duplicates()

pairs_year_df = pairs_df.merge(paper_year, on="paper_id", how="left")


pairs_year_df = pairs_year_df[pairs_year_df["author_1_id"].isin(gender_lookup.keys()) &
    pairs_year_df["author_2_id"].isin(gender_lookup.keys())].copy()

pairs_year_df["gender_1"] = pairs_year_df["author_1_id"].map(gender_lookup)
pairs_year_df["gender_2"] = pairs_year_df["author_2_id"].map(gender_lookup)

pairs_year_df = pairs_year_df[pairs_year_df["gender_1"].isin(["female", "male"]) &
    pairs_year_df["gender_2"].isin(["female", "male"])].copy()

In [ ]:
def yearly_gender_bias(df_year, B=50, seed=42):

    edges_year = (df_year.groupby(["author_1_id", "author_2_id"], as_index=False)
        .size().rename(columns={"size": "weight"}))

    G_year = nx.Graph()

    for _, row in edges_year.iterrows():
        G_year.add_edge(row["author_1_id"],row["author_2_id"],weight=row["weight"])


    nx.set_node_attributes(G_year,
        {n: gender_lookup[n] for n in G_year.nodes() if n in gender_lookup},
        "gender")

    keep_nodes = [n for n, d in G_year.nodes(data=True)
        if d.get("gender") in ["female", "male"]]
    G_year = G_year.subgraph(keep_nodes).copy()

    if G_year.number_of_edges() == 0:
        return {
            "num_nodes": G_year.number_of_nodes(),
            "num_edges": G_year.number_of_edges(),
            "female_share": np.nan,
            "assortativity": np.nan,
            "same_gender_share_obs": np.nan,
            "same_gender_share_null": np.nan,
            "same_gender_excess_pp": np.nan,
            "assortativity_z": np.nan}

    genders = np.array([G_year.nodes[n]["gender"] for n in G_year.nodes()])
    female_share = (genders == "female").mean()

    assort_obs = nx.attribute_assortativity_coefficient(G_year, "gender")

    same_obs = sum(1 for u, v in G_year.edges()
        if G_year.nodes[u]["gender"] == G_year.nodes[v]["gender"]) / G_year.number_of_edges()

    rng = np.random.default_rng(seed)
    nodes = list(G_year.nodes())

    assort_null = []
    same_null = []

    for _ in range(B):
        shuffled = rng.permutation(genders)

        G_rand = G_year.copy()
        nx.set_node_attributes(G_rand,
            {node: shuffled[i] for i, node in enumerate(nodes)},
            "gender")

        assort_null.append(nx.attribute_assortativity_coefficient(G_rand, "gender"))

        same_null.append(sum(
                1 for u, v in G_rand.edges()
                if G_rand.nodes[u]["gender"] == G_rand.nodes[v]["gender"]) / G_rand.number_of_edges())

    assort_null = np.array(assort_null)
    same_null = np.array(same_null)

    assort_sd = assort_null.std(ddof=1)

    return {
        "num_nodes": G_year.number_of_nodes(),
        "num_edges": G_year.number_of_edges(),
        "female_share": female_share,
        "assortativity": assort_obs,
        "same_gender_share_obs": same_obs,
        "same_gender_share_null": same_null.mean(),
        "same_gender_excess_pp": 100 * (same_obs - same_null.mean()),
        "assortativity_z": np.nan if assort_sd == 0 else (assort_obs - assort_null.mean()) / assort_sd}

In [ ]:
results = []

for year in range(2021, 2026):
    df_year = pairs_year_df[pairs_year_df["publication_year"] == year].copy()
    stats = yearly_gender_bias(df_year, B=100, seed=year)
    stats["year"] = year
    results.append(stats)

bias_by_year = pd.DataFrame(results).sort_values("year").reset_index(drop=True)


bias_table = bias_by_year.copy()

bias_table["female_share_pct"] = 100 * bias_table["female_share"]
bias_table["same_gender_share_obs_pct"] = 100 * bias_table["same_gender_share_obs"]
bias_table["same_gender_share_null_pct"] = 100 * bias_table["same_gender_share_null"]

bias_table = bias_table[[
        "year",
        "num_nodes",
        "num_edges",
        "female_share_pct",
        "assortativity",
        "assortativity_z",
        "same_gender_share_obs_pct",
        "same_gender_share_null_pct",
        "same_gender_excess_pp"]].round(3)

bias_table

,year,num_nodes,num_edges,female_share_pct,assortativity,assortativity_z,same_gender_share_obs_pct,same_gender_share_null_pct,same_gender_excess_pp
0,2021,51209,99654,37.911,0.147,47.975,59.310,52.919,6.391
1,2022,49034,95757,37.813,0.157,48.005,59.492,52.990,6.503
2,2023,51920,102930,37.596,0.146,50.373,59.101,53.111,5.991
3,2024,54722,116241,39.352,0.144,45.707,58.263,52.286,5.977
4,2025,49644,106458,39.874,0.136,44.274,57.681,52.059,5.622
